# 🚀 Bane Agent: Neural DPO Benchmark (Google Colab T4 GPU + Google Drive)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariveshkoshta-spec/Bane_Agent/blob/main/Bane_Colab_GPU_Benchmark.ipynb)

This turnkey notebook runs our fine-tuned DPO LLaMA-3 model (`bane_dpo_lora_adapters`) with **full hardware acceleration** on an NVIDIA T4 GPU.
It loads your LoRA adapters directly from **Google Drive** and benchmarks all **30 Enterprise Queries** (20 Technical + 10 Conversational Slang) against `enterprise_nexus.sqlite`.

### Step 1: Verify GPU & Fast Install (~25s)

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ GPU NOT ACTIVE! Go to Runtime -> Change runtime type -> select 'T4 GPU' -> Save"
print(f"✅ GPU Active: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")

# Fast install pre-built wheels without compiling from source (avoids RAM/CPU spikes)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets
!pip install faiss-cpu sentence-transformers rich tabulate
print("\n✅ All GPU dependencies installed!")

### Step 2: Clone Bane Agent & Prepare Database

In [ ]:
import os
if not os.path.exists("/content/Bane_Agent") and not os.path.exists("Bane_Agent"):
    !git clone https://github.com/pariveshkoshta-spec/Bane_Agent.git

if os.path.exists("/content/Bane_Agent"):
    %cd /content/Bane_Agent
elif os.path.exists("Bane_Agent"):
    %cd Bane_Agent

!git pull

# Ensure enterprise_nexus database is ready
if not os.path.exists("enterprise_nexus.sqlite"):
    !python scripts/generate_enterprise_nexus.py

print(f"✅ Enterprise Database Ready: {os.path.exists('enterprise_nexus.sqlite')}")

### Step 3: Mount Google Drive & Auto-Detect Adapters 📂
Mounts your Google Drive, finds `bane_dpo_lora_adapters` (or `.zip`), and copies it to local fast SSD storage.

In [ ]:
import os
import shutil
import glob
import zipfile

# 1. Mount Google Drive
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print("⏳ Requesting Google Drive access... click 'Connect to Google Drive' when prompted:")
        drive.mount('/content/drive')
    print("✅ Google Drive successfully mounted!")
except Exception as e:
    print(f"[INFO] Drive mount skipped or not on Colab: {e}")

# 2. Locate adapters folder or zip file in Drive
found_adapter = None

drive_candidates = [
    "/content/drive/MyDrive/bane_dpo_lora_adapters",
    "/content/drive/MyDrive/Bane_Agent/results/bane_dpo_lora_adapters",
    "/content/drive/MyDrive/results/bane_dpo_lora_adapters",
    "/content/drive/MyDrive/Colab Notebooks/bane_dpo_lora_adapters",
    "/content/bane_dpo_lora_adapters",
    "results/bane_dpo_lora_adapters"
]

for dc in drive_candidates:
    if os.path.exists(dc) and os.path.exists(os.path.join(dc, "adapter_config.json")):
        found_adapter = dc
        break

# Check for zip in Google Drive if folder not directly found
if not found_adapter and os.path.exists("/content/drive/MyDrive"):
    zip_matches = glob.glob("/content/drive/MyDrive/**/bane_dpo_lora_adapters*.zip", recursive=True)
    if zip_matches:
        print(f"📦 Found adapter zip in Google Drive: {zip_matches[0]}. Extracting...")
        os.makedirs("/content/bane_dpo_lora_adapters", exist_ok=True)
        with zipfile.ZipFile(zip_matches[0], 'r') as zip_ref:
            zip_ref.extractall("/content/bane_dpo_lora_adapters")
        found_adapter = "/content/bane_dpo_lora_adapters"

# Recursive search in Drive if still not found
if not found_adapter and os.path.exists("/content/drive/MyDrive"):
    print("🔍 Searching your Google Drive for 'adapter_config.json'...")
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        depth = root.count(os.sep) - "/content/drive/MyDrive".count(os.sep)
        if depth > 4: # limit depth for speed
            continue
        if "adapter_config.json" in files:
            found_adapter = root
            break

# 3. Copy to local high-speed SSD (/content/bane_dpo_lora_adapters)
local_adapter_dir = "/content/bane_dpo_lora_adapters"
if found_adapter:
    print(f"📁 Located adapters at: {found_adapter}")
    if os.path.abspath(found_adapter) != os.path.abspath(local_adapter_dir):
        print("⚡ Copying adapters to local SSD for ultra-fast GPU loading...")
        os.makedirs(local_adapter_dir, exist_ok=True)
        for item in os.listdir(found_adapter):
            src = os.path.join(found_adapter, item)
            dst = os.path.join(local_adapter_dir, item)
            if os.path.isfile(src):
                shutil.copy2(src, dst)
        print(f"✅ Successfully staged adapters to: {local_adapter_dir}")
    adapter_path = local_adapter_dir
else:
    adapter_path = None
    print("❌ Could not find 'bane_dpo_lora_adapters' in Google Drive.")
    print("👉 Please upload the 'bane_dpo_lora_adapters' folder (or zip) into your Google Drive, then re-run this cell.")

### Step 4: Load LLaMA-3 + LoRA Weights into 16GB GPU VRAM (~25s)

In [ ]:
from unsloth import FastLanguageModel

load_target = adapter_path if adapter_path else "unsloth/llama-3-8b-Instruct"
print(f"⏳ Loading model from: {load_target}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = load_target,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print(f"🚀 Fine-Tuned Model Successfully Loaded into GPU VRAM!")

### Step 5: Run Full 30-Question Benchmark at GPU Speed (~1s per query) 🚀

In [ ]:
import sqlite3
import time
import json
from src.db_introspector import DatabaseIntrospector
from src.schema_rag import SchemaRetriever
from src.prompt_builder import format_dpo_prompt
from scripts.evaluate_part_a import test_suite_part_a
from scripts.evaluate_part_b import part_b_questions

# 1. Initialize FAISS schema index
introspector = DatabaseIntrospector("enterprise_nexus.sqlite")
schemas = introspector.extract_schemas()
retriever = SchemaRetriever()
retriever.build_index(schemas)
print(f"✅ FAISS indexed {len(schemas)} enterprise tables into vector space.\n")

# 2. Prepare 30 benchmark queries
all_questions = []
for q in test_suite_part_a:
    all_questions.append({
        "id": q["id"],
        "title": q["title"],
        "question": q["question"],
        "expected_sql": q["expected_sql"],
        "type": "Part A (Technical)"
    })
for q in part_b_questions:
    all_questions.append({
        "id": q["id"],
        "title": q["title"],
        "question": q["question"],
        "expected_sql": q["expected_sql"],
        "type": "Part B (Conversational)"
    })

db_conn = sqlite3.connect("enterprise_nexus.sqlite")
cursor = db_conn.cursor()
results = []

print(f"🔥 Evaluating {len(all_questions)} Queries on NVIDIA T4 GPU...\n" + "="*75)

for item in all_questions:
    q_id = item["id"]
    q_text = item["question"]
    exp_sql = item["expected_sql"].strip()

    # RAG schema retrieval
    context = retriever.retrieve_context(q_text, top_k=3)
    prompt = format_dpo_prompt(q_text, context)

    # Neural inference on GPU
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = time.time() - t0
    raw_sql = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    gen_sql = raw_sql.replace(prompt, "").strip()

    # Clean markdown formatting
    if "```sql" in gen_sql:
        gen_sql = gen_sql.split("```sql")[1].split("```")[0].strip()
    elif "```" in gen_sql:
        gen_sql = gen_sql.split("```")[1].split("```")[0].strip()

    # Execute on enterprise_nexus.sqlite
    exec_status = "SUCCESS"
    err_str = None
    gen_rows = []
    try:
        cursor.execute(gen_sql)
        gen_rows = cursor.fetchall()
    except Exception as e:
        exec_status = "EXEC_ERROR"
        err_str = str(e)

    # Execute expected
    cursor.execute(exp_sql)
    exp_rows = cursor.fetchall()

    icon = "✅" if exec_status == "SUCCESS" else "❌"
    print(f"[{icon}] Q{q_id:02d} [{item['type']}]: {item['title']} ({latency:.2f}s | {len(gen_rows)} rows)")
    if exec_status != "SUCCESS":
        print(f"      Error: {err_str}")
        print(f"      SQL:   {gen_sql[:100]}...")

    results.append({
        "id": q_id,
        "title": item["title"],
        "type": item["type"],
        "question": q_text,
        "agent_sql": gen_sql,
        "expected_sql": exp_sql,
        "status": exec_status,
        "error": err_str,
        "gen_rows": len(gen_rows),
        "exp_rows": len(exp_rows),
        "latency": round(latency, 2)
    })

db_conn.close()

# Save results
with open("colab_gpu_results.json", "w") as f:
    json.dump(results, f, indent=2)

total = len(results)
pass_count = sum(1 for r in results if r["status"] == "SUCCESS")
print("\n" + "="*75)
print(f"🎉 Complete! Pass Rate: {pass_count}/{total} ({pass_count/total*100:.1f}%)")

### Step 6: Render Comprehensive Scorecard & Query Review Table

In [ ]:
from IPython.display import display, Markdown

part_a_succ = sum(1 for r in results if "Part A" in r["type"] and r["status"] == "SUCCESS")
part_b_succ = sum(1 for r in results if "Part B" in r["type"] and r["status"] == "SUCCESS")

md_scorecard = f"""
## 📊 GPU Neural Benchmark Scorecard

| Section | Total Queries | Syntax Execution Pass | Accuracy Rate |
| :--- | :---: | :---: | :---: |
| **Part A: Technical & Analytical** | 20 | **{part_a_succ}/20** | **{part_a_succ/20*100:.1f}%** |
| **Part B: Conversational Slang** | 10 | **{part_b_succ}/10** | **{part_b_succ/10*100:.1f}%** |
| **TOTAL OVERALL** | **30** | **{pass_count}/30** | **{pass_count/30*100:.1f}%** |
"""
display(Markdown(md_scorecard))

# Show first 5 queries side by side
print("\nSample Query Outputs:")
for r in results[:5]:
    display(Markdown(f"### Q{r['id']}: {r['title']}\n**Question:** *{r['question']}*\n\n**Generated SQL:**\n```sql\n{r['agent_sql']}\n```\n**Target SQL:**\n```sql\n{r['expected_sql']}\n```"))